# 04 - Feature engineering

Builds and caches the three feature families and fuses them:

| Family | Prefix | Source |
|---|---|---|
| Acoustic | `aud_` | librosa on the waveform |
| Text / linguistic | `txt_`, `ling_` | rule-based features from the transcript |
| Embeddings | `emb_` | MiniLM sentence embedding (PCA is fitted *inside CV folds*, never here) |

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)

In [ ]:
RUN_FEATURES = False   # True runs extraction here; the CLI `python scripts/extract_features.py` does the same
if RUN_FEATURES and ctx is not None:
    from src.pipeline import run_audio_features, run_text_features, run_embeddings, build_merged
    for split in ("train", "test"):
        run_audio_features(ctx, split); run_text_features(ctx, split); run_embeddings(ctx, split); build_merged(ctx, split)

In [ ]:
mp = ctx.processed("merged_features") / "train_features.csv" if ctx else None
if mp is not None and mp.exists():
    df = pd.read_csv(mp)
    from src.features.fusion import group_columns
    for g in ("audio", "text", "embeddings"):
        print(f"{g:11s}: {len(group_columns(df.columns, [g]))} columns")
    print("rows:", len(df), "| NaN cells in tabular features:", int(df[group_columns(df.columns, ['audio','text'])].isna().sum().sum()))
else:
    print(PENDING)

## Which handcrafted features move with the label? (Spearman, exploratory)

In [ ]:
if mp is not None and mp.exists():
    from src.features.fusion import group_columns
    tab = group_columns(df.columns, ["audio", "text"])
    corr = df[tab].corrwith(df["label"], method="spearman").dropna().sort_values(key=abs, ascending=False)
    display(corr.head(20).to_frame("spearman"))
    corr.head(20).iloc[::-1].plot.barh(figsize=(7, 6), title="Top |Spearman| with label"); plt.show()

## Embedding dimensionality (exploratory only)
Fitting PCA on the full training set here is for *looking* at the variance spectrum. The models never use this PCA: `GroupFeatureBuilder` refits PCA on each training fold.

In [ ]:
if mp is not None and mp.exists():
    from sklearn.decomposition import PCA
    E = df[group_columns(df.columns, ["embeddings"])].to_numpy()
    if E.size:
        p = PCA(n_components=min(100, E.shape[1], E.shape[0] - 1)).fit(E)
        plt.plot(np.cumsum(p.explained_variance_ratio_)); plt.axvline(cfg["features"]["pca_components"], color="r", ls="--")
        plt.xlabel("components"); plt.ylabel("cumulative explained variance"); plt.title("Embedding PCA (exploratory)"); plt.show()